<a href="https://colab.research.google.com/github/nick1982ad/CS_Data_analysis/blob/main/Jupyter%20Notebooks/pygrok_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Парсинг событий безопасности с помощью шаблонов Grok в среде Python

## Библиотека `pygrok`

---

**Grok** — это язык шаблонов для разбора неструктурированного текста (логов, событий, сообщений) в структурированные данные. Изначально Grok создан [Jordan Sissel](https://github.com/jordansissel/grok) и широко используется в **Logstash** (ELK-стек), **Fluentd**, **Telegraf** и других инструментах обработки событий.

Ключевая идея: вместо написания сложных регулярных выражений вручную мы собираем шаблон из **именованных строительных блоков** — предопределённых паттернов, каждый из которых скрывает за собой регулярное выражение (иногда очень сложное).

**`pygrok`** — наиболее зрелая реализация Grok в Python. Это *не обёртка над оригиналом*, а полностью самостоятельная реализация.

### Полезные ссылки

| Ресурс | URL |
|--------|-----|
| pygrok на GitHub | https://github.com/garyelephant/pygrok |
| pygrok на PyPI | https://pypi.org/project/pygrok/ |
| Исходники паттернов Logstash | https://github.com/logstash-plugins/logstash-patterns-core/tree/main/patterns |
| Документация Grok-фильтра Logstash | https://www.elastic.co/guide/en/logstash/current/plugins-filters-grok.html |
| Grok Debugger (онлайн) | https://grokdebugger.com/ |
| Grok Constructor (онлайн) | https://grokconstructor.appspot.com/ |

### Содержание

1. [Установка и первый запуск](#1)
2. [Синтаксис Grok-шаблонов](#2)
3. [Каталог встроенных шаблонов](#3)
4. [Приведение типов](#4)
5. [Частичное совпадение и отладка](#5)
6. [Разбор реальных форматов событий](#6)
   - 6.1 Syslog (RFC 3164)
   - 6.2 Apache / Nginx access log
   - 6.3 SSH auth log
   - 6.4 Cisco ASA firewall
   - 6.5 JSON-обёрнутые события
7. [Создание собственных шаблонов](#7)
   - 7.1 Inline-шаблоны
   - 7.2 Словарь `custom_patterns`
   - 7.3 Директория с файлами шаблонов
8. [Пакетный разбор и интеграция с Pandas](#8)
9. [Разбор событий auditd (Linux)](#9)
10. [Разбор событий безопасности Windows](#10)
11. [Заключение](#11)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


<a id='1'></a>
## 1. Установка и первый запуск

`pygrok` зависит от модуля **`regex`** (расширенная альтернатива стандартному `re`), который поддерживает атомарные группы `(?>...)`, необходимую для ряда Grok-паттернов.

Установка: `pip install pygrok` (модуль `regex` установится автоматически).

> **Документация pygrok:** полный README и примеры использования — https://github.com/garyelephant/pygrok

In [2]:
# Установка (раскомментируйте при необходимости)
!pip install pygrok

  Preparing metadata (setup.py) ... done
  Created wheel for pygrok: filename=pygrok-1.0.0-py3-none-any.whl size=21725 sha256=9200c3719fc137b60590599c21a3d675d244fc5272c1889ea6aff83bcbda71ac
  Stored in directory: /root/.cache/pip/wheels/61/36/5d/7cbf2201c0bcddcb3b5eb5bc24324d1e938c360aaf3ffb5762
Successfully built pygrok


В первом примере мы используем три встроенных шаблона. Ниже приведено их описание:

| Паттерн | Regex | Описание |
|---------|-------|----------|
| `IP` | `(?:%{IPV6}\|%{IPV4})` | IPv4 или IPv6 адрес. Составной паттерн — объединяет `IPV4` и `IPV6` через операцию логического "или" |
| `IPV4` | `(?<![0-9])(?:(?:[0-1]?[0-9]{1,2}\|2[0-4][0-9]\|25[0-5])[.]...){3}(?![0-9])` | IPv4-адрес с проверкой границ (lookbehind/lookahead предотвращают ложные срабатывания внутри чисел) |
| `WORD` | `\b\w+\b` | Одно слово — последовательность буквенно-цифровых символов и символа `_`, ограниченная границами слова |
| `URIPATHPARAM` | `%{URIPATH}(?:%{URIPARAM})?` | Путь URI с необязательной query-строкой, напр. `/api/v2/users?active=true` |
| `URIPATH` | `(?:/[A-Za-z0-9$.+!*'(){},~:;=@#%_\-]*)+` | Путь URI (включающий один или несколько `/`-сегментов) |
| `URIPARAM` | `\?[A-Za-z0-9$.+!*'\|(){},~@#%&/=:;_?\-\[\]<>]*` | Query string (начиная с `?`) |

In [3]:
from pygrok import Grok

# Простейший пример: извлечение IP-адреса и метода HTTP-запроса
pattern = '%{IP:client_ip} %{WORD:method} %{URIPATHPARAM:request}'
text = '192.168.1.42 GET /api/v2/users?active=true'

grok = Grok(pattern)
result = grok.match(text)
print(result)

{'client_ip': '192.168.1.42', 'method': 'GET', 'request': '/api/v2/users?active=true'}


In [4]:
# Если хоть немного изменить последовательность символов в событии, то такой шаблон уже не сработает
text = '192.168.1.42 GET  /api/v2/users?active=true'

result = grok.match(text)
print(result)

None


Результат — обычный Python `dict`, готовый для дальнейшей обработки.

Если строка не соответствует шаблону, `match()` вернёт `None`:

<a id='2'></a>
## 2. Синтаксис Grok-шаблонов

Основная единица — конструкция вида:

```
%{PATTERN_NAME:field_name}
```

| Компонент | Описание |
|-----------|----------|
| `PATTERN_NAME` | Имя предопределённого паттерна (по сути, именованное регулярное выражение) |
| `field_name` | Имя поля (ключ результирующего словаря), в которое будет сохранено извлечённое значение |

Между конструкциями `%{...}` можно вставлять **литеральный текст** и **произвольные regex**:
Пример:
```
%{IP:src} -> %{IP:dst} \[%{WORD:action}\]
```

Если имя поля не нужно (требуется совпадение, но не захват в переменную), можно использовать только имя паттерна:

```
%{SPACE}%{WORD:status}
```

Однако, для составных шаблонов поля могут быть предопределены в файле конфигурации

> **Полный синтаксис Grok** описан в документации Logstash: https://www.elastic.co/guide/en/logstash/current/plugins-filters-grok.html

Новые паттерны, используемые в этом разделе:

| Паттерн | Regex | Описание |
|---------|-------|----------|
| `SPACE` | `\s*` | Ноль или более пробельных символов (пробел, табуляция, перенос строки) |
| `GREEDYDATA` | `.*` | «Жадный» захват — максимально возможное количество любых символов до конца строки |
| `DATA` | `.*?` | «Ленивый» (нежадный) захват — минимально возможное количество символов |
| `NOTSPACE` | `\S+` | Один или более непробельных символов |

> **`GREEDYDATA` vs `DATA`** — ключевое различие. `DATA` (`.*?`) останавливается при первой возможности, `GREEDYDATA` (`.*`) захватывает всё до конца. Используйте `DATA` в середине шаблона (между другими паттернами) и `GREEDYDATA` только в конце.

In [5]:
# Пример: разбор строки межсетевого экрана
fw_pattern = '%{IP:src} -> %{IP:dst} proto=%{WORD:proto} action=%{WORD:action}'
fw_line = '10.0.0.5 -> 192.168.1.1 proto=TCP action=DENY'

grok_fw = Grok(fw_pattern)
print(grok_fw.match(fw_line))

{'src': '10.0.0.5', 'dst': '192.168.1.1', 'proto': 'TCP', 'action': 'DENY'}


In [6]:
# Паттерн без захвата — %{PATTERN} без :field_name
# Здесь SPACE используется просто для пропуска пробелов
pattern_no_capture = '%{SPACE}%{WORD:level} %{GREEDYDATA:message}'
grok_nc = Grok(pattern_no_capture)
print(grok_nc.match('   ERROR disk quota exceeded'))

{'level': 'ERROR', 'message': 'disk quota exceeded'}


In [7]:
# И теперь использование %{SPACE} позволяет менять начало события (добавлять пробельные символы, включая переносы строк)
print(grok_nc.match("""
    ERROR disk quota exceeded"""))

{'level': 'ERROR', 'message': 'disk quota exceeded'}


<a id='3'></a>
## 3. Каталог встроенных шаблонов

`pygrok` поставляется с обширной библиотекой шаблонов проекта Logstash. Давайте посмотрим, какие файлы и паттерны доступны.

> **Актуальная коллекция шаблонов напрямую из Grok** (может быть новее, чем в pygrok): https://github.com/logstash-plugins/logstash-patterns-core/tree/main/patterns/ecs-v1


In [8]:
import os
import pygrok

patterns_dir = os.path.join(os.path.dirname(pygrok.__file__), 'patterns')
print('Директория паттернов:', patterns_dir)
print()



Директория паттернов: /usr/local/lib/python3.12/dist-packages/pygrok/patterns



In [9]:
for fname in sorted(os.listdir(patterns_dir)):
    fpath = os.path.join(patterns_dir, fname)
    # Считаем количество шаблонов в каждом файле
    with open(fpath) as f:
        count = sum(1 for line in f if line.strip() and not line.startswith('#'))
    print(f'  {fname:25s} — {count} шаблон(ов)')

  aws                       — 6 шаблон(ов)
  bacula                    — 47 шаблон(ов)
  bro                       — 4 шаблон(ов)
  exim                      — 12 шаблон(ов)
  firewalls                 — 44 шаблон(ов)
  grok-patterns             — 76 шаблон(ов)
  haproxy                   — 7 шаблон(ов)
  java                      — 13 шаблон(ов)
  junos                     — 4 шаблон(ов)
  linux-syslog              — 10 шаблон(ов)
  mcollective               — 1 шаблон(ов)
  mcollective-patterns      — 2 шаблон(ов)
  mongodb                   — 7 шаблон(ов)
  nagios                    — 61 шаблон(ов)
  postgresql                — 1 шаблон(ов)
  rails                     — 7 шаблон(ов)
  redis                     — 2 шаблон(ов)
  ruby                      — 2 шаблон(ов)


In [10]:
fpath = os.path.join(patterns_dir, "grok-patterns")
with open(fpath) as f:
    for line in f:
        if line.strip() and not line.startswith('#'):
            print(line)


USERNAME [a-zA-Z0-9._-]+

USER %{USERNAME}

EMAILLOCALPART [a-zA-Z][a-zA-Z0-9_.+-=:]+

EMAILADDRESS %{EMAILLOCALPART}@%{HOSTNAME}

HTTPDUSER %{EMAILADDRESS}|%{USER}

INT (?:[+-]?(?:[0-9]+))

BASE10NUM (?<![0-9.+-])(?>[+-]?(?:(?:[0-9]+(?:\.[0-9]+)?)|(?:\.[0-9]+)))

NUMBER (?:%{BASE10NUM})

BASE16NUM (?<![0-9A-Fa-f])(?:[+-]?(?:0x)?(?:[0-9A-Fa-f]+))

BASE16FLOAT \b(?<![0-9A-Fa-f.])(?:[+-]?(?:0x)?(?:(?:[0-9A-Fa-f]+(?:\.[0-9A-Fa-f]*)?)|(?:\.[0-9A-Fa-f]+)))\b

POSINT \b(?:[1-9][0-9]*)\b

NONNEGINT \b(?:[0-9]+)\b

WORD \b\w+\b

NOTSPACE \S+

SPACE \s*

DATA .*?

GREEDYDATA .*

QUOTEDSTRING (?>(?<!\\)(?>"(?>\\.|[^\\"]+)+"|""|(?>'(?>\\.|[^\\']+)+')|''|(?>`(?>\\.|[^\\`]+)+`)|``))

UUID [A-Fa-f0-9]{8}-(?:[A-Fa-f0-9]{4}-){3}[A-Fa-f0-9]{12}

MAC (?:%{CISCOMAC}|%{WINDOWSMAC}|%{COMMONMAC})

CISCOMAC (?:(?:[A-Fa-f0-9]{4}\.){2}[A-Fa-f0-9]{4})

WINDOWSMAC (?:(?:[A-Fa-f0-9]{2}-){5}[A-Fa-f0-9]{2})

COMMONMAC (?:(?:[A-Fa-f0-9]{2}:){5}[A-Fa-f0-9]{2})

IPV6 ((([0-9A-Fa-f]{1,4}:){7}([0-9A-Fa-f]{1,4}|:))|(([0-

### Справочник ключевых шаблонов

Ниже справочная таблица наиболее важных шаблонов из файла `grok-patterns`. Для составных паттернов (обозначенных через `%{...}`) указана Grok-формула; для атомарных — regex.

#### Текст и общие захваты

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `WORD` | `\b\w+\b` | Одно слово (буквы, цифры, `_`) |
| `NOTSPACE` | `\S+` | Один и более непробельных символов |
| `SPACE` | `\s*` | Ноль и более пробельных символов |
| `DATA` | `.*?` | Минимальный (ленивый) захват произвольного текста |
| `GREEDYDATA` | `.*` | Максимальный (жадный) захват до конца строки |
| `QUOTEDSTRING` / `QS` | Атомарные группы для `"..."`, `'...'`, `` `...` `` | Строка в кавычках (с учётом экранирования) |

#### Числа

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `NUMBER` | `(?:%{BASE10NUM})` | Целое или дробное число (`42`, `-3.14`, `+0.5`) |
| `BASE10NUM` | `(?<![0-9.+-])(?>[+-]?(?:(?:[0-9]+(?:\.[0-9]+)?)\|(?:\.[0-9]+)))` | Десятичное число с опциональным знаком и дробной частью |
| `INT` | `(?:[+-]?(?:[0-9]+))` | Целое число со знаком |
| `POSINT` | `\b(?:[1-9][0-9]*)\b` | Строго положительное целое число (≥ 1) |
| `NONNEGINT` | `\b(?:[0-9]+)\b` | Неотрицательное целое число (≥ 0) |
| `BASE16NUM` | `(?<![0-9A-Fa-f])(?:[+-]?(?:0x)?(?:[0-9A-Fa-f]+))` | Шестнадцатеричное число |

#### Сетевые адреса и идентификаторы

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `IP` | `(?:%{IPV6}\|%{IPV4})` | IPv4 или IPv6 адрес |
| `IPV4` | `(?<![0-9])(?:[0-1]?[0-9]{1,2}\|2[0-4][0-9]\|25[0-5])[.]...(?![0-9])` | IPv4-адрес с граничными проверками |
| `IPV6` | RFC-совместимый regex (~600 символов) | Полный IPv6 адрес (все формы записи) |
| `HOSTNAME` | `\b(?:[0-9A-Za-z][0-9A-Za-z-]{0,62})(?:\.(...)){*}` | DNS-имя хоста |
| `IPORHOST` | `(?:%{IP}\|%{HOSTNAME})` | IP-адрес или hostname |
| `HOSTPORT` | `%{IPORHOST}:%{POSINT}` | Хост с портом (`server:8080`) |
| `MAC` | `(?:%{CISCOMAC}\|%{WINDOWSMAC}\|%{COMMONMAC})` | MAC-адрес (Cisco / Windows / Unix форматы) |
| `USERNAME` / `USER` | `[a-zA-Z0-9._-]+` | Имя пользователя |
| `EMAILADDRESS` | `%{EMAILLOCALPART}@%{HOSTNAME}` | Email-адрес |
| `UUID` | `[A-Fa-f0-9]{8}-(?:[A-Fa-f0-9]{4}-){3}[A-Fa-f0-9]{12}` | UUID / GUID |

#### URI и пути файловой системы

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `URI` | `%{URIPROTO}://(?:%{USER}(?::[^@]*)?@)?(?:%{URIHOST})?(?:%{URIPATHPARAM})?` | Полный URI |
| `URIPATH` | `(?:/[A-Za-z0-9$.+!*'(){},~:;=@#%_\-]*)+` | Путь в URI |
| `URIPATHPARAM` | `%{URIPATH}(?:%{URIPARAM})?` | Путь + query string |
| `PATH` | `(?:%{UNIXPATH}\|%{WINPATH})` | Путь ФС (Unix или Windows) |
| `UNIXPATH` | `(/[\w_%!$@:.,~-]+\|\\\\.)+ ` | Unix-путь (`/var/log/syslog`) |
| `WINPATH` | `(?>[A-Za-z]+:\|\\\\)(?:\\\\[^\\\\?*]*)+` | Windows-путь (`C:\Windows\System32`) |

#### Дата и время

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `TIMESTAMP_ISO8601` | `%{YEAR}-%{MONTHNUM}-%{MONTHDAY}[T ]%{HOUR}:?%{MINUTE}(?::?%{SECOND})?%{ISO8601_TIMEZONE}?` | дата в формате ISO 8601 (`2025-03-15T10:23:45+03:00`) |
| `SYSLOGTIMESTAMP` | `%{MONTH} +%{MONTHDAY} %{TIME}` | Формат syslog (`Mar 15 10:23:45`) |
| `HTTPDATE` | `%{MONTHDAY}/%{MONTH}/%{YEAR}:%{TIME} %{INT}` | Формат Apache (`15/Mar/2025:10:23:45 +0300`) |
| `DATE_US` / `DATE_EU` | `%{MONTHNUM}[/-]%{MONTHDAY}[/-]%{YEAR}` / `%{MONTHDAY}[./-]%{MONTHNUM}[./-]%{YEAR}` | Даты US/EU формата |
| `TIME` | `%{HOUR}:%{MINUTE}(?::%{SECOND})` | Время `HH:MM:SS` |
| `MONTH` | `Jan(?:uary)?`, `Feb`, ... `Dec(?:ember)?` | Название месяца (полное или сокращённое) |
| `YEAR` | `(\d\d){1,2}` | Год (2 или 4 цифры) |

#### Журналы событий

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `LOGLEVEL` | `Alert\|Trace\|Debug\|Notice\|Info\|Warn(?:ing)?\|Err(?:or)?\|Crit(?:ical)?\|Fatal\|Severe\|Emerg(?:ency)?` | Уровень важности (регистронезависимый) |


<a id='4'></a>
## 4. Приведение типов

По умолчанию все извлечённые значения — строки. Однако `pygrok` поддерживает автоматическое приведение типов с помощью расширенного синтаксиса:

```
%{PATTERN:field_name:type}
```

Поддерживаемые типы: `int` и `float`

In [11]:
# Без приведения типов — всё строки
pattern_str = '%{IP:ip} %{NUMBER:bytes} %{NUMBER:duration}'
text = '10.0.0.1 15824 0.043'

result_str = Grok(pattern_str).match(text)
print('Без типизации:', result_str)
print('  bytes:', type(result_str['bytes']), result_str['bytes'])

Без типизации: {'ip': '10.0.0.1', 'bytes': '15824', 'duration': '0.043'}
  bytes: <class 'str'> 15824


In [12]:
# С приведением типов
pattern_typed = '%{IP:ip} %{NUMBER:bytes:int} %{NUMBER:duration:float}'
result_typed = Grok(pattern_typed).match(text)
print('С типизацией: ', result_typed)
print('  bytes:', type(result_typed['bytes']), result_typed['bytes'])
print('  duration:', type(result_typed['duration']), result_typed['duration'])

С типизацией:  {'ip': '10.0.0.1', 'bytes': 15824, 'duration': 0.043}
  bytes: <class 'int'> 15824
  duration: <class 'float'> 0.043


Типизация особенно полезна при дальнейшей аналитике — не нужно вручную конвертировать `response_code`, `bytes_sent` и т.д.

<a id='5'></a>
## 5. Частичное совпадение и отладка

Важная особенность `pygrok`: метод `match()` выполняет **частичное совпадение** — паттерн может совпасть с подстрокой входной строки. Несовпавшие части в начале и конце игнорируются.

> Это поведение отличается от Logstash, где Grok по умолчанию ожидает совпадение всей строки. В `pygrok` для полного совпадения нужно явно добавить стандартные якоря регулярных выражений `^` и `$`.

In [13]:
# Частичное совпадение: паттерн совпадает с серединой строки
grok_partial = Grok('%{IP:addr}')

# Здесь IP окружён другим текстом, но совпадающая часть будет корректно распознана
result = grok_partial.match('Клиент с адресом 192.168.1.100 отключился')
print('Частичное совпадение:', result)

# Если нужно ПОЛНОЕ совпадение — привязываем к началу/концу через regex
grok_full = Grok('^%{IP:addr}$')
print('Полное (IP):        ', grok_full.match('192.168.1.100'))
print('Полное (с текстом): ', grok_full.match('Клиент 192.168.1.100'))

Частичное совпадение: {'addr': '192.168.1.100'}
Полное (IP):         {'addr': '192.168.1.100'}
Полное (с текстом):  None


### Пошаговая отладка шаблона

> **Совет**: собирайте шаблон инкрементально, проверяя `match()` после добавления каждого нового элемента. Для интерактивной отладки также удобен [Grok Debugger](https://grokdebugger.com/).

In [14]:
# Приём для отладки: строим шаблон постепенно
# Есть событие, которое нужно разобрать:
log_line = '2025-03-15T10:23:45+03:00 webserver01 nginx: 192.168.1.50 "GET /index.html" 200 1234'

# Шаг 1: начинаем с временной метки
g1 = Grok('%{TIMESTAMP_ISO8601:ts}')
print('Шаг 1:', g1.match(log_line))

# Шаг 2: добавляем hostname
g2 = Grok('%{TIMESTAMP_ISO8601:ts} %{HOSTNAME:host}')
print('Шаг 2:', g2.match(log_line))

# Шаг 3: программа
g3 = Grok('%{TIMESTAMP_ISO8601:ts} %{HOSTNAME:host} %{WORD:program}:')
print('Шаг 3:', g3.match(log_line))

# Шаг 4: полный шаблон
g4 = Grok('%{TIMESTAMP_ISO8601:ts} %{HOSTNAME:host} %{WORD:program}: %{IP:client} "%{WORD:method} %{URIPATH:uri}" %{NUMBER:status:int} %{NUMBER:bytes:int}')
print('Шаг 4:', g4.match(log_line))

Шаг 1: {'ts': '2025-03-15T10:23:45+03:00'}
Шаг 2: {'ts': '2025-03-15T10:23:45+03:00', 'host': 'webserver01'}
Шаг 3: {'ts': '2025-03-15T10:23:45+03:00', 'host': 'webserver01', 'program': 'nginx'}
Шаг 4: {'ts': '2025-03-15T10:23:45+03:00', 'host': 'webserver01', 'program': 'nginx', 'client': '192.168.1.50', 'method': 'GET', 'uri': '/index.html', 'status': 200, 'bytes': 1234}


> Можно также использовать инкрементальный построитель шаблонов Grok: [Grok Constructor](https://grokconstructor.appspot.com/do/construction)

> **Совет по отладке**: собирайте шаблон инкрементально, проверяя `match()` после добавления каждого нового элемента. Так вы сразу увидите, где шаблон «ломается».

### 6.1 Syslog (RFC 3164)

Для разбора syslog-событий `pygrok` предоставляет семейство готовых составных шаблонов:

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `SYSLOGTIMESTAMP` | `%{MONTH} +%{MONTHDAY} %{TIME}` | Метка времени syslog: месяц, день, время (`Mar 15 10:23:45`) |
| `SYSLOGHOST` | `%{IPORHOST}` | Хост-источник — IP-адрес или hostname |
| `SYSLOGPROG` | `%{PROG:program}(?:\[%{POSINT:pid}\])?` | Имя программы с необязательным PID в скобках (`sshd[12345]`) |
| `PROG` | `[\x21-\x5a\x5c\x5e-\x7e]+` | Имя программы (печатные ASCII-символы кроме `[`) |
| `POSINT` | `\b(?:[1-9][0-9]*)\b` | Строго положительное целое число |
| `SYSLOGBASE` | `%{SYSLOGTIMESTAMP:timestamp} (?:%{SYSLOGFACILITY} )?%{SYSLOGHOST:logsource} %{SYSLOGPROG}:` | Полный заголовок syslog |

> **Формат syslog** определён в [RFC 3164](https://datatracker.ietf.org/doc/html/rfc3164) (BSD syslog) и [RFC 5424](https://datatracker.ietf.org/doc/html/rfc5424) (современный). `pygrok` поддерживает оба.

In [15]:
import json

def pp(result):
    """Pretty-print результата разбора."""
    if result is None:
        print('  ❌ Совпадение не найдено')
    else:
        for k, v in result.items():
            print(f'  {k:20s} = {v!r}')

In [16]:
# Syslog RFC 3164: встроенный шаблон SYSLOGLINE
syslog_samples = [
    'Mar 15 10:23:45 webserver01 sshd[12345]: Accepted publickey for admin from 10.0.0.5 port 52413 ssh2',
    'Mar 15 10:24:01 dbserver02 CRON[9876]: (root) CMD (/usr/sbin/ntpdate pool.ntp.org)',
    'Mar 15 10:25:12 firewall01 kernel: [UFW BLOCK] IN=eth0 OUT= SRC=203.0.113.50 DST=10.0.0.1 PROTO=TCP SPT=44321 DPT=22',
]

grok_syslog = Grok('%{SYSLOGTIMESTAMP:timestamp} %{SYSLOGHOST:host} %{SYSLOGPROG}: %{GREEDYDATA:message}')

for line in syslog_samples:
    print(f'Исходное событие: {line[:70]}...')
    pp(grok_syslog.match(line))
    print()

Исходное событие: Mar 15 10:23:45 webserver01 sshd[12345]: Accepted publickey for admin ...
  timestamp            = 'Mar 15 10:23:45'
  host                 = 'webserver01'
  program              = 'sshd'
  pid                  = '12345'
  message              = 'Accepted publickey for admin from 10.0.0.5 port 52413 ssh2'

Исходное событие: Mar 15 10:24:01 dbserver02 CRON[9876]: (root) CMD (/usr/sbin/ntpdate p...
  timestamp            = 'Mar 15 10:24:01'
  host                 = 'dbserver02'
  program              = 'CRON'
  pid                  = '9876'
  message              = '(root) CMD (/usr/sbin/ntpdate pool.ntp.org)'

Исходное событие: Mar 15 10:25:12 firewall01 kernel: [UFW BLOCK] IN=eth0 OUT= SRC=203.0....
  timestamp            = 'Mar 15 10:25:12'
  host                 = 'firewall01'
  program              = 'kernel'
  pid                  = None
  message              = '[UFW BLOCK] IN=eth0 OUT= SRC=203.0.113.50 DST=10.0.0.1 PROTO=TCP SPT=44321 DPT=22'



### 6.2 Apache / Nginx access log

Для Apache-логов в pygrok есть два готовых составных паттерна: COMMONAPACHELOG и COMBINEDAPACHELOG

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `COMMONAPACHELOG` | `%{IPORHOST:clientip} %{HTTPDUSER:ident} %{USER:auth} \[%{HTTPDATE:timestamp}\] "..." %{NUMBER:response} ...` | Common Log Format (CLF): клиент, аутентификация, запрос, код ответа, размер |
| `COMBINEDAPACHELOG` | `%{COMMONAPACHELOG} %{QS:referrer} %{QS:agent}` | Combined Log: CLF + Referer + User-Agent |
| `HTTPDATE` | `%{MONTHDAY}/%{MONTH}/%{YEAR}:%{TIME} %{INT}` | Формат даты Apache (`15/Mar/2025:10:23:45 +0300`) |
| `QS` / `QUOTEDSTRING` | Атомарные группы для `"..."`, `'...'`, `` `...` `` | Строка в кавычках (с экранированием `\"`) |
| `IPORHOST` | `(?:%{IP}\|%{HOSTNAME})` | IP-адрес или DNS-имя хоста |

> **Формат Combined Log** описан в [документации Apache](https://httpd.apache.org/docs/current/logs.html#combined).

In [17]:
# Apache Combined Log Format — встроенный шаблон COMBINEDAPACHELOG
apache_lines = [
    '192.168.1.100 - admin [15/Mar/2025:10:23:45 +0300] "GET /dashboard HTTP/1.1" 200 5432 "https://example.com/login" "Mozilla/5.0 (X11; Linux x86_64)"',
    '10.0.0.5 - - [15/Mar/2025:10:24:01 +0300] "POST /api/data HTTP/1.1" 403 128 "-" "curl/7.68.0"',
]

grok_apache = Grok('%{COMBINEDAPACHELOG}')

for line in apache_lines:
    print(f'Исходное событие: {line[:60]}...')
    pp(grok_apache.match(line))
    print()

Исходное событие: 192.168.1.100 - admin [15/Mar/2025:10:23:45 +0300] "GET /das...
  clientip             = '192.168.1.100'
  ident                = '-'
  auth                 = 'admin'
  timestamp            = '15/Mar/2025:10:23:45 +0300'
  verb                 = 'GET'
  request              = '/dashboard'
  httpversion          = '1.1'
  rawrequest           = None
  response             = '200'
  bytes                = '5432'
  referrer             = '"https://example.com/login"'
  agent                = '"Mozilla/5.0 (X11; Linux x86_64)"'

Исходное событие: 10.0.0.5 - - [15/Mar/2025:10:24:01 +0300] "POST /api/data HT...
  clientip             = '10.0.0.5'
  ident                = '-'
  auth                 = '-'
  timestamp            = '15/Mar/2025:10:24:01 +0300'
  verb                 = 'POST'
  request              = '/api/data'
  httpversion          = '1.1'
  rawrequest           = None
  response             = '403'
  bytes                = '128'
  referrer             = '"-"

Для **Nginx error log** готового паттерна нет, но его легко составить из базовых блоков.

In [19]:
# Nginx error log — кастомный шаблон
nginx_error = '2025/03/15 10:30:22 [error] 1234#0: *5678 open() "/var/www/html/robots.txt" failed (2: No such file or directory), client: 192.168.1.50, server: example.com, request: "GET /robots.txt HTTP/1.1"'
nginx_error_wo_cid = '2025/03/15 10:30:22 [error] 1234#0: open() "/var/www/html/robots.txt" failed (2: No such file or directory), client: 192.168.1.50, server: example.com, request: "GET /robots.txt HTTP/1.1"'

#части шаблона могут быть опциональными (но внимательнее с пробелами!)
nginx_pattern = ('%{DATA:timestamp} \[%{WORD:severity}\] '
                 '%{NUMBER:pid}#%{NUMBER:tid}:(?: \*%{NUMBER:connection_id})? '
                 '%{GREEDYDATA:message}, client: %{IPORHOST:client}, '
                 'server: %{IPORHOST:server}, '
                 'request: "%{WORD:method} %{URIPATHPARAM:uri} HTTP/%{NUMBER:http_version}"')

print('Nginx error log:')
pp(Grok(nginx_pattern).match(nginx_error))
print('Nginx error log:')
pp(Grok(nginx_pattern).match(nginx_error_wo_cid))


Nginx error log:
  timestamp            = '2025/03/15 10:30:22'
  severity             = 'error'
  pid                  = '1234'
  tid                  = '0'
  connection_id        = '5678'
  message              = 'open() "/var/www/html/robots.txt" failed (2: No such file or directory)'
  client               = '192.168.1.50'
  server               = 'example.com'
  method               = 'GET'
  uri                  = '/robots.txt'
  http_version         = '1.1'
Nginx error log:
  timestamp            = '2025/03/15 10:30:22'
  severity             = 'error'
  pid                  = '1234'
  tid                  = '0'
  connection_id        = None
  message              = 'open() "/var/www/html/robots.txt" failed (2: No such file or directory)'
  client               = '192.168.1.50'
  server               = 'example.com'
  method               = 'GET'
  uri                  = '/robots.txt'
  http_version         = '1.1'


<>:6: SyntaxWarning: invalid escape sequence '\['
<>:7: SyntaxWarning: invalid escape sequence '\*'
<>:6: SyntaxWarning: invalid escape sequence '\['
<>:7: SyntaxWarning: invalid escape sequence '\*'
/tmp/ipykernel_3740/1926599902.py:6: SyntaxWarning: invalid escape sequence '\['
  nginx_pattern = ('%{DATA:timestamp} \[%{WORD:severity}\] '
/tmp/ipykernel_3740/1926599902.py:7: SyntaxWarning: invalid escape sequence '\*'
  '%{NUMBER:pid}#%{NUMBER:tid}:(?: \*%{NUMBER:connection_id})? '


### 6.3 SSH auth log

In [20]:
ssh_events = [
    'Mar 15 10:23:45 server01 sshd[12345]: Accepted password for user1 from 10.0.0.5 port 52413 ssh2',
    'Mar 15 10:23:50 server01 sshd[12346]: Failed password for invalid user root from 203.0.113.50 port 44321 ssh2',
    'Mar 15 10:24:00 server01 sshd: Accepted publickey for admin from 10.0.0.10 port 60001 ssh2: RSA SHA256:abc123',
    'Mar 15 10:24:05 server01 sshd[12348]: Connection closed by authenticating user testuser 192.168.1.200 port 55555 [preauth]',
    'Mar 15 10:24:05 server01 sshd: Connection closed by remote peer 192.168.15.2 port 55555',
    'Mar 15 10:25:01 server01 cron[14152]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)'
]

# Шаблон для успешной/неуспешной аутентификации
# Уже нелья использовать SYSLOGPROG, так как матчиться должны только строки с именем процесса sshd
ssh_auth_pattern = ('%{SYSLOGTIMESTAMP:timestamp} %{HOSTNAME:host} sshd(?:\[%{POSINT:pid}\])?: '
                    '%{WORD:result} %{WORD:auth_method} for (%{DATA:invalid_flag} )?%{USERNAME:user} '
                    'from %{IP:src_ip} port %{POSINT:src_port:int} %{WORD:protocol}')

grok_ssh = Grok(ssh_auth_pattern)

for line in ssh_events:
    result = grok_ssh.match(line)
    if result:
        status = '✅' if result.get('result') == 'Accepted' else '❌'
        invalid = ' (INVALID USER)' if result.get('invalid_flag') else ''
        print(f'{status} {result["user"]}{invalid} от {result["src_ip"]}:{result["src_port"]} '
              f'метод={result["auth_method"]}')
    else:
        # Не подходит под шаблон аутентификации
        print(f'⚠️  Не распознано: {line[25:70]}...')

✅ user1 от 10.0.0.5:52413 метод=password
❌ root (INVALID USER) от 203.0.113.50:44321 метод=password
✅ admin от 10.0.0.10:60001 метод=publickey
⚠️  Не распознано: sshd[12348]: Connection closed by authenticat...
⚠️  Не распознано: sshd: Connection closed by remote peer 192.16...
⚠️  Не распознано: cron[14152]: (root) CMD (cd / && run-parts --...


<>:12: SyntaxWarning: invalid escape sequence '\['
<>:12: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_3740/466932673.py:12: SyntaxWarning: invalid escape sequence '\['
  ssh_auth_pattern = ('%{SYSLOGTIMESTAMP:timestamp} %{HOSTNAME:host} sshd(?:\[%{POSINT:pid}\])?: '


### 6.4 Cisco ASA firewall

`pygrok` включает набор готовых паттернов для Cisco ASA (файл `firewalls`). Основные:

| Паттерн | Определение | Описание |
|---------|-------------|----------|
| `CISCOTIMESTAMP` | `%{MONTH} +%{MONTHDAY}(?: %{YEAR})? %{TIME}` | Метка времени Cisco (год необязателен) |
| `CISCOTAG` | `[A-Z0-9]+-%{INT}-(?:[A-Z0-9_]+)` | Тег сообщения Cisco (`ASA-6-302013`): facility-severity-mnemonic |
| `CISCO_ACTION` | `Built\|Teardown\|Deny\|Denied\|...` | Действие МСЭ: список конкретных ключевых слов |
| `CISCO_DIRECTION` | `Inbound\|inbound\|Outbound\|outbound` | Направление трафика |

> **Полный набор Cisco-паттернов** (30+ правил для конкретных ASA message ID) — в файле `firewalls` директории паттернов pygrok или в [исходниках Logstash](https://github.com/logstash-plugins/logstash-patterns-core/blob/main/patterns/ecs-v1/firewalls).

In [21]:
# Cisco ASA-6-302013: Built TCP connection
asa_line = 'Mar 15 2025 10:23:45: %ASA-6-302013: Built inbound TCP connection 12345 for outside:203.0.113.50/44321 (203.0.113.50/44321) to inside:10.0.0.5/443 (10.0.0.5/443)'

# Разбираем заголовок + тело отдельно
asa_header = '%{CISCOTIMESTAMP:timestamp}: %%{CISCOTAG:tag}:'
asa_body = '%{CISCO_ACTION:action}(?: %{CISCO_DIRECTION:direction})? %{WORD:protocol} connection %{INT:conn_id} for %{DATA:src_iface}:%{IP:src_ip}/%{INT:src_port}'

grok_asa = Grok(asa_header + ' ' + asa_body)
result = grok_asa.match(asa_line)
print('Cisco ASA event:')
pp(result)

Cisco ASA event:
  timestamp            = 'Mar 15 2025 10:23:45'
  tag                  = 'ASA-6-302013'
  action               = 'Built'
  direction            = 'inbound'
  protocol             = 'TCP'
  conn_id              = '12345'
  src_iface            = 'outside'
  src_ip               = '203.0.113.50'
  src_port             = '44321'


### 6.5 Комбинация с предварительной обработкой (JSON-обёрнутые события)

Иногда события приходят уже частично форматированные (например, в формате JSON), тогда grok может обогатить исходное событие ("допарсить")

In [22]:
import json

# Событие из SIEM в JSON-формате
json_event = '''{
    "source": "proxy-01",
    "category": "web",
    "raw": "192.168.1.100 - admin [15/Mar/2025:10:23:45 +0300] \\"GET /admin/settings HTTP/1.1\\" 200 2048 \\"https://intranet.corp.local/\\" \\"Mozilla/5.0\\""
}'''

event = json.loads(json_event)
print(f'Источник: {event["source"]}, категория: {event["category"]}')

# Применяем Grok к полю raw
parsed = Grok('%{COMBINEDAPACHELOG}').match(event['raw'])

# Обогащаем исходное событие
enriched = {**event, **parsed}
#del enriched['raw']  # Убираем сырые данные
print('Обогащённое событие:')
print(json.dumps(enriched, indent=2, ensure_ascii=False))

Источник: proxy-01, категория: web
Обогащённое событие:
{
  "source": "proxy-01",
  "category": "web",
  "raw": "192.168.1.100 - admin [15/Mar/2025:10:23:45 +0300] \"GET /admin/settings HTTP/1.1\" 200 2048 \"https://intranet.corp.local/\" \"Mozilla/5.0\"",
  "clientip": "192.168.1.100",
  "ident": "-",
  "auth": "admin",
  "timestamp": "15/Mar/2025:10:23:45 +0300",
  "verb": "GET",
  "request": "/admin/settings",
  "httpversion": "1.1",
  "rawrequest": null,
  "response": "200",
  "bytes": "2048",
  "referrer": "\"https://intranet.corp.local/\"",
  "agent": "\"Mozilla/5.0\""
}


<a id='7'></a>
## 7. Создание собственных шаблонов

Встроенных паттернов не всегда достаточно. `pygrok` предоставляет три способа определения собственных паттернов.

> **Документация:** способы задания кастомных паттернов описаны в [README pygrok](https://github.com/garyelephant/pygrok#getting-started) и в [документации Logstash Grok](https://www.elastic.co/guide/en/logstash/current/plugins-filters-grok.html#_custom_patterns).

### 7.1 Inline — regex прямо в шаблоне

Можно вставить произвольное регулярное выражение напрямую.
В этом случае определяется именованная группа регулярного выражения `(?<name>выражение)`

In [23]:
# Inline regex для разбора событий Suricata EVE (упрощённо)
suricata_alert = '[**] [1:2100498:7] GPL ATTACK_RESPONSE id check returned root [**] [Classification: Potentially Bad Traffic] [Priority: 2] 03/15/2025-10:23:45.123456 10.0.0.5:80 -> 192.168.1.100:44321'

# Используем inline regex для SID и GID
suricata_pattern = ('\[\*\*\] \[(?<gid>\d+):(?<sid>\d+):(?<rev>\d+)\] '
                    '%{DATA:signature} \[\*\*\] '
                    '\[Classification: %{DATA:classification}\] '
                    '\[Priority: %{INT:priority:int}\] '
                    '%{DATA:timestamp} '
                    '%{IP:src_ip}:%{POSINT:src_port:int} -> %{IP:dst_ip}:%{POSINT:dst_port:int}')

result = Grok(suricata_pattern).match(suricata_alert)
print('Suricata alert:')
pp(result)

Suricata alert:
  gid                  = '1'
  sid                  = '2100498'
  rev                  = '7'
  signature            = 'GPL ATTACK_RESPONSE id check returned root'
  classification       = 'Potentially Bad Traffic'
  priority             = 2
  timestamp            = '03/15/2025-10:23:45.123456'
  src_ip               = '10.0.0.5'
  src_port             = 80
  dst_ip               = '192.168.1.100'
  dst_port             = 44321


<>:5: SyntaxWarning: invalid escape sequence '\['
<>:6: SyntaxWarning: invalid escape sequence '\['
<>:7: SyntaxWarning: invalid escape sequence '\['
<>:8: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\['
<>:6: SyntaxWarning: invalid escape sequence '\['
<>:7: SyntaxWarning: invalid escape sequence '\['
<>:8: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_3740/2345777783.py:5: SyntaxWarning: invalid escape sequence '\['
  suricata_pattern = ('\[\*\*\] \[(?<gid>\d+):(?<sid>\d+):(?<rev>\d+)\] '
/tmp/ipykernel_3740/2345777783.py:6: SyntaxWarning: invalid escape sequence '\['
  '%{DATA:signature} \[\*\*\] '
/tmp/ipykernel_3740/2345777783.py:7: SyntaxWarning: invalid escape sequence '\['
  '\[Classification: %{DATA:classification}\] '
/tmp/ipykernel_3740/2345777783.py:8: SyntaxWarning: invalid escape sequence '\['
  '\[Priority: %{INT:priority:int}\] '


### 7.2 Словарь `custom_patterns`

Можно передать в конструктор словарь с определениями новых паттернов, которые затем доступны в шаблоне как `%{PATTERN_NAME}`:

In [24]:
# Определяем кастомные паттерны для разбора Snort/Suricata fast.log
custom = {
    'SNORT_TIMESTAMP':  r'\d{2}/\d{2}/\d{4}-\d{2}:\d{2}:\d{2}\.\d+',
    'SNORT_SIG':        r'\[\d+:\d+:\d+\]',
    'SNORT_PRIORITY':   r'\d',
    'PROTO':            r'TCP|UDP|ICMP|GRE',
}

snort_line = '03/15/2025-10:23:45.654321  [**] [1:2001219:20] ET SCAN Potential SSH Scan [**] [Classification: Attempted Information Leak] [Priority: 2] {TCP} 203.0.113.50:44321 -> 10.0.0.5:22'

snort_pattern = ('%{SNORT_TIMESTAMP:timestamp}  \[\*\*\] %{SNORT_SIG:sig_id} '
                 '%{DATA:signature} \[\*\*\] '
                 '\[Classification: %{DATA:classification}\] '
                 '\[Priority: %{SNORT_PRIORITY:priority}\] '
                 '\{%{PROTO:protocol}\} '
                 '%{IP:src_ip}:%{POSINT:src_port:int} -> %{IP:dst_ip}:%{POSINT:dst_port:int}')



#в переменной custom_patterns передается словарь-расширение
grok_snort = Grok(snort_pattern, custom_patterns=custom)
result = grok_snort.match(snort_line)
print('Snort/Suricata fast.log:')
pp(result)

Snort/Suricata fast.log:
  timestamp            = '03/15/2025-10:23:45.654321'
  sig_id               = '[1:2001219:20]'
  signature            = 'ET SCAN Potential SSH Scan'
  classification       = 'Attempted Information Leak'
  priority             = '2'
  protocol             = 'TCP'
  src_ip               = '203.0.113.50'
  src_port             = 44321
  dst_ip               = '10.0.0.5'
  dst_port             = 22


<>:11: SyntaxWarning: invalid escape sequence '\['
<>:12: SyntaxWarning: invalid escape sequence '\['
<>:13: SyntaxWarning: invalid escape sequence '\['
<>:14: SyntaxWarning: invalid escape sequence '\['
<>:15: SyntaxWarning: invalid escape sequence '\{'
<>:11: SyntaxWarning: invalid escape sequence '\['
<>:12: SyntaxWarning: invalid escape sequence '\['
<>:13: SyntaxWarning: invalid escape sequence '\['
<>:14: SyntaxWarning: invalid escape sequence '\['
<>:15: SyntaxWarning: invalid escape sequence '\{'
/tmp/ipykernel_3740/3731706253.py:11: SyntaxWarning: invalid escape sequence '\['
  snort_pattern = ('%{SNORT_TIMESTAMP:timestamp}  \[\*\*\] %{SNORT_SIG:sig_id} '
/tmp/ipykernel_3740/3731706253.py:12: SyntaxWarning: invalid escape sequence '\['
  '%{DATA:signature} \[\*\*\] '
/tmp/ipykernel_3740/3731706253.py:13: SyntaxWarning: invalid escape sequence '\['
  '\[Classification: %{DATA:classification}\] '
/tmp/ipykernel_3740/3731706253.py:14: SyntaxWarning: invalid escape sequence '\['
 

### 7.3 Директория с файлами паттернов (`custom_patterns_dir`)

Для крупных проектов удобнее хранить определения в файлах. Формат файла:
```
# Комментарий
PATTERN_NAME regex
```

In [25]:
import os, tempfile

# Создаём временную директорию с нашими паттернами
patterns_tmpdir = tempfile.mkdtemp(prefix='grok_patterns_')

# Файл с паттернами для CEF (Common Event Format)
cef_patterns = """# CEF (Common Event Format) patterns
CEF_VERSION CEF:\\d+
CEF_SEVERITY (?:[0-9]|10)
CEF_HEADER %{CEF_VERSION}\\|%{DATA:device_vendor}\\|%{DATA:device_product}\\|%{DATA:device_version}\\|%{DATA:sig_id}\\|%{DATA:name}\\|%{CEF_SEVERITY:severity}\\|
"""

with open(os.path.join(patterns_tmpdir, 'cef'), 'w') as f:
    f.write(cef_patterns)

print(f'Файлы паттернов в {patterns_tmpdir}:')
for fname in os.listdir(patterns_tmpdir):
    print(f'  {fname}')
    with open(os.path.join(patterns_tmpdir, fname)) as f:
        print(f'  Содержимое:\n{f.read()}')

Файлы паттернов в /tmp/grok_patterns_tubu76nz:
  cef
  Содержимое:
# CEF (Common Event Format) patterns
CEF_VERSION CEF:\d+
CEF_SEVERITY (?:[0-9]|10)
CEF_HEADER %{CEF_VERSION}\|%{DATA:device_vendor}\|%{DATA:device_product}\|%{DATA:device_version}\|%{DATA:sig_id}\|%{DATA:name}\|%{CEF_SEVERITY:severity}\|



In [26]:
# Теперь используем эти паттерны
cef_event = 'CEF:0|Fortinet|FortiGate|7.0.5|0419016384|SSL connection established|3|src=10.0.0.5 dst=203.0.113.50 spt=44321 dpt=443 proto=TCP'

# Заголовок CEF разбираем кастомным паттерном, расширение — отдельно
cef_full_pattern = '%{CEF_HEADER}%{GREEDYDATA:extension}'

#параметр custom_patterns_dir передает путь к каталогу с нашими кастомными определенями
grok_cef = Grok(cef_full_pattern, custom_patterns_dir=patterns_tmpdir)
result = grok_cef.match(cef_event)
print('CEF event (заголовок):')
pp(result)

# Дополнительно разбираем поле extension (key=value пары) - с помощью обычного регулярного выражения
if result and result.get('extension'):
    import re
    ext_pairs = dict(re.findall(r'(\w+)=([^\s]+)', result['extension']))
    print('\nCEF extension (разобранное):')
    for k, v in ext_pairs.items():
        print(f'  {k:10s} = {v}')

CEF event (заголовок):
  device_vendor        = 'Fortinet'
  device_product       = 'FortiGate'
  device_version       = '7.0.5'
  sig_id               = '0419016384'
  name                 = 'SSL connection established'
  severity             = '3'
  extension            = 'src=10.0.0.5 dst=203.0.113.50 spt=44321 dpt=443 proto=TCP'

CEF extension (разобранное):
  src        = 10.0.0.5
  dst        = 203.0.113.50
  spt        = 44321
  dpt        = 443
  proto      = TCP


In [27]:
# Очистка
import shutil
shutil.rmtree(patterns_tmpdir)

<a id='8'></a>
## 8. Пакетный разбор и интеграция с Pandas

На практике требуется обрабатывать тысячи и даже сотни тысяч событий за раз. `pygrok` хорошо сочетается с Pandas для таких задач.

> **Важно**: объект `Grok()` компилирует regex при создании. При пакетной обработке создавайте его **один раз** и переиспользуйте для всех строк.

In [28]:
import pandas as pd
from pygrok import Grok

# Имитация лога Apache
access_log = [
    '192.168.1.100 - admin [15/Mar/2025:10:00:01 +0300] "GET /index.html HTTP/1.1" 200 5432 "-" "Mozilla/5.0"',
    '192.168.1.101 - - [15/Mar/2025:10:00:02 +0300] "POST /api/login HTTP/1.1" 401 128 "-" "curl/7.68.0"',
    '10.0.0.5 - svc_account [15/Mar/2025:10:00:03 +0300] "GET /api/data HTTP/1.1" 200 102400 "-" "python-requests/2.28"',
    '203.0.113.50 - - [15/Mar/2025:10:00:04 +0300] "GET /admin/../../../etc/passwd HTTP/1.1" 400 0 "-" "Nikto/2.1"',
    '192.168.1.100 - admin [15/Mar/2025:10:00:05 +0300] "GET /dashboard HTTP/1.1" 200 8192 "https://example.com/" "Mozilla/5.0"',
    '10.0.0.5 - svc_account [15/Mar/2025:10:00:06 +0300] "DELETE /api/users/42 HTTP/1.1" 204 0 "-" "python-requests/2.28"',
    '203.0.113.50 - - [15/Mar/2025:10:00:07 +0300] "GET /wp-admin/install.php HTTP/1.1" 404 0 "-" "Nikto/2.1"',
]

grok_combined = Grok('%{COMBINEDAPACHELOG}')

# Парсим все строки, отфильтровывая несовпавшие
parsed_rows = []
for line in access_log:
    result = grok_combined.match(line)
    if result:
        parsed_rows.append(result)

df = pd.DataFrame(parsed_rows)

# Приводим типы
df['response'] = df['response'].astype(int)
df['bytes'] = df['bytes'].astype(int)

print(f'Разобрано {len(df)} из {len(access_log)} строк\n')
df[['clientip', 'auth', 'verb', 'request', 'response', 'bytes', 'agent']]

Разобрано 7 из 7 строк



,clientip,auth,verb,request,response,bytes,agent
0,192.168.1.100,admin,GET,/index.html,200,5432,"""Mozilla/5.0"""
1,192.168.1.101,-,POST,/api/login,401,128,"""curl/7.68.0"""
2,10.0.0.5,svc_account,GET,/api/data,200,102400,"""python-requests/2.28"""
3,203.0.113.50,-,GET,/admin/../../../etc/passwd,400,0,"""Nikto/2.1"""
4,192.168.1.100,admin,GET,/dashboard,200,8192,"""Mozilla/5.0"""
5,10.0.0.5,svc_account,DELETE,/api/users/42,204,0,"""python-requests/2.28"""
6,203.0.113.50,-,GET,/wp-admin/install.php,404,0,"""Nikto/2.1"""


In [30]:
#Проанализируем настоящий лог файл
f = open('/content/drive/MyDrive/DataAnalysis/promjetDec2021.log')
access_log = f.read().splitlines()
len(access_log)

137510

In [31]:
parsed_rows = []
for line in access_log:
    result = grok_combined.match(line)
    if result:
        parsed_rows.append(result)

df = pd.DataFrame(parsed_rows)

# Приводим типы
df['response'] = df['response'].astype(int)
#df['bytes'] = df['bytes'].astype(int)

print(f'Разобрано {len(df)} из {len(access_log)} строк\n')
df[['clientip', 'auth', 'verb', 'request', 'response', 'bytes', 'agent']]

Разобрано 137510 из 137510 строк



,clientip,auth,verb,request,response,bytes,agent
0,63.143.42.249,-,GET,/,200,18648,"""Mozilla/5.0+(compatible; UptimeRobot/2.0; htt..."
1,185.103.167.218,-,GET,/jet/company/875.html,404,70695,"""Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/5..."
2,63.143.42.247,-,HEAD,/,200,None,"""Mozilla/5.0+(compatible; UptimeRobot/2.0; htt..."
3,93.84.69.87,-,GET,/favicon.ico,200,None,"""Mozilla/5.0 (Windows NT 5.1; rv:52.0) Gecko/2..."
4,93.84.69.87,-,GET,/favicon.ico,200,None,"""Mozilla/5.0 (Windows NT 5.1; rv:52.0) Gecko/2..."
...,...,...,...,...,...,...,...
137505,63.143.42.247,-,HEAD,/,200,None,"""Mozilla/5.0+(compatible; UptimeRobot/2.0; htt..."
137506,95.73.173.17,-,HEAD,/,200,None,"""Xenu Link Sleuth/1.3.8"""
137507,63.143.42.249,-,GET,/,200,18649,"""Mozilla/5.0+(compatible; UptimeRobot/2.0; htt..."
137508,77.88.5.51,-,GET,/wp-content/uploads/2014/07/21.jpg,200,3942,"""Mozilla/5.0 (compatible; YandexImages/3.0; +h..."


In [32]:
# Аналитика: подозрительная активность
print('=== Статистика по кодам ответа ===')
print(df['response'].value_counts().to_string())

=== Статистика по кодам ответа ===
response
200    126568
404      7244
301      2047
304       525
403       445
307       273
206       245
406       131
503        13
400        10
302         6
401         3


In [33]:
print('\n=== Запросы с ошибками (5xx) по IP ===')
df[df['response'] >= 500].head()







=== Запросы с ошибками (5xx) по IP ===


,clientip,ident,auth,timestamp,verb,request,httpversion,rawrequest,response,bytes,referrer,agent
19800,171.22.180.9,-,-,04/Dec/2021:17:24:30 +0300,GET,/shop/ventzip/%D1%88%D1%82%D1%83%D1%86%D0%B5%D...,1.1,None,503,428,"""-""","""Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/5..."
19881,185.244.217.232,-,-,04/Dec/2021:18:06:33 +0300,GET,/,1.1,None,503,79156,"""-""","""Mozilla/5.0 (Linux; Android 10; Mi 9T) AppleW..."
22076,45.66.13.135,-,-,05/Dec/2021:06:24:29 +0300,GET,/shop/medikal_equipment/%D0%B7%D0%BC%D0%B5%D0%...,1.1,None,503,428,"""-""","""Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/5..."
31262,63.143.42.249,-,-,07/Dec/2021:06:43:58 +0300,GET,/,1.1,None,503,79156,"""http://promjet.ru""","""Mozilla/5.0+(compatible; UptimeRobot/2.0; htt..."
44102,95.217.122.213,-,-,09/Dec/2021:18:32:17 +0300,GET,/,1.1,None,503,79156,"""-""","""Mozilla/5.0 (Windows NT 10.0; WOW64; rv:45.0)..."


In [34]:
print('\n=== Запросы с ошибками (403) по IP ===')
df[df['response'] == 403].head()



=== Запросы с ошибками (403) по IP ===


,clientip,ident,auth,timestamp,verb,request,httpversion,rawrequest,response,bytes,referrer,agent
2353,35.243.255.156,-,-,01/Dec/2021:04:14:31 +0300,GET,/,1.0,None,403,79156,"""-""","""ZoominfoBot (zoominfobot at zoominfo dot com)"""
2354,35.243.255.156,-,-,01/Dec/2021:04:14:32 +0300,GET,/,1.0,None,403,79156,"""-""","""ZoominfoBot (zoominfobot at zoominfo dot com)"""
2355,35.243.255.156,-,-,01/Dec/2021:04:14:32 +0300,GET,/,1.0,None,403,79156,"""-""","""ZoominfoBot (zoominfobot at zoominfo dot com)"""
2356,35.243.255.156,-,-,01/Dec/2021:04:14:32 +0300,GET,/,1.0,None,403,79156,"""-""","""ZoominfoBot (zoominfobot at zoominfo dot com)"""
2357,35.243.255.156,-,-,01/Dec/2021:04:14:30 +0300,GET,/robots.txt,1.0,None,403,689,"""-""","""ZoominfoBot (zoominfobot at zoominfo dot com)"""


In [35]:
print(df[df['response'] == 403]['clientip'].value_counts().to_string())

clientip
35.243.255.156     60
34.75.110.192      36
104.196.114.245    24
35.231.71.243      24
216.244.66.250     16
34.75.95.73        12
34.139.253.149     12
35.237.69.66       12
35.229.90.101      12
34.74.205.224      12
213.87.138.245     10
5.9.144.234         6
144.76.186.38       6
148.251.120.201     6
144.76.118.82       6
192.99.37.116       4
75.119.142.113      4
148.251.9.145       4
178.63.87.197       4
185.234.69.220      4
144.76.96.236       4
144.76.120.197      4
144.76.137.254      4
35.229.38.177       4
157.230.230.140     3
195.123.209.118     3
95.173.161.74       3
192.99.10.170       2
107.174.235.110     2
192.99.13.75        2
144.76.91.79        2
5.9.138.189         2
148.251.69.139      2
85.10.207.195       2
5.9.155.37          2
91.121.66.88        2
161.97.135.248      2
173.249.7.244       2
192.99.36.166       2
75.119.158.122      2
144.76.38.40        2
213.239.216.194     2
167.114.159.183     2
144.76.176.171      2
185.234.69.215      2
1

In [36]:
print('\n=== Запросы с ошибками (401) по IP ===')
df[df['response'] == 401]


=== Запросы с ошибками (401) по IP ===


,clientip,ident,auth,timestamp,verb,request,httpversion,rawrequest,response,bytes,referrer,agent
97254,95.214.54.97,-,-,21/Dec/2021:07:42:41 +0300,GET,/wp-json/wp/v2/users/3,1.1,None,401,402,"""http://promjet.ru/""","""Mozilla/5.0 (iPhone; CPU iPhone OS 11_3 like ..."
97255,185.220.100.240,-,-,21/Dec/2021:07:42:41 +0300,GET,/wp-json/wp/v2/users/4,1.1,None,401,402,"""http://promjet.ru/""","""Mozilla/5.0 (Windows; U; Windows NT 5.1; en-U..."
97256,209.141.59.243,-,-,21/Dec/2021:07:42:42 +0300,GET,/wp-json/wp/v2/users/5,1.1,None,401,402,"""http://promjet.ru/""","""Mozilla/5.0 (Windows NT 5.1) AppleWebKit/537...."


<a id='9'></a>
## 9. Разбор событий auditd (Linux)

**auditd** — подсистема аудита ядра Linux. Её события имеют характерный формат:
```
type=TYPE msg=audit(EPOCH:SERIAL): key=value key=value ...
```

Основная сложность — у событий auditd произвольный набор key=value пар, которые зависят от типа события. Стратегия: **Grok для заголовка** + **Python regex для key=value пар** в теле.

Для заголовка мы определим собственные паттерны:

| Кастомный паттерн | Regex | Описание |
|-------------------|-------|----------|
| `AUDIT_TYPE` | `[A-Z_]+` | Тип события auditd (`SYSCALL`, `AVC`, `USER_AUTH`, ...) |
| `AUDIT_EPOCH` | `\d+\.\d+` | Unix-время с дробной частью (секунды.микросекунды) |
| `AUDIT_SERIAL` | `\d+` | Серийный номер события (для группировки связанных записей) |

| Используемый встроенный паттерн | Определение | Описание |
|---------------------------------|-------------|----------|
| `NONNEGINT` | `\b(?:[0-9]+)\b` | Неотрицательное целое число (для uid, gid, pid, ...) |

> **Справка по auditd:** типы записей описаны в `man auditd(8)` и в [документации Red Hat](https://access.redhat.com/documentation/en-us/red_hat_enterprise_linux/7/html/security_guide/sec-audit_record_types).

In [37]:
# Примеры реальных событий auditd
auditd_events = [
    # SYSCALL: выполнение команды
    'type=SYSCALL msg=audit(1710494625.123:1001): arch=c000003e syscall=59 success=yes exit=0 a0=55a1b2c3d4e5 a1=55a1b2c3d4f0 a2=55a1b2c3d500 a3=0 items=2 ppid=1234 pid=5678 auid=1000 uid=0 gid=0 euid=0 suid=0 fsuid=0 egid=0 sgid=0 fsgid=0 tty=pts0 ses=3 comm="passwd" exe="/usr/bin/passwd" key="privileged_commands"',

    # EXECVE: аргументы команды
    'type=EXECVE msg=audit(1710494625.123:1001): argc=3 a0="/usr/bin/passwd" a1="-l" a2="testuser"',

    # USER_AUTH: аутентификация
    'type=USER_AUTH msg=audit(1710494630.456:1002): pid=5679 uid=0 auid=1000 ses=3 msg=\'op=PAM:authentication grantors=pam_unix acct="root" exe="/usr/bin/sudo" hostname=? addr=? terminal=/dev/pts/0 res=success\'',

    # PATH: информация о пути к файлу
    'type=PATH msg=audit(1710494625.123:1001): item=0 name="/usr/bin/passwd" inode=262278 dev=08:01 mode=0104755 ouid=0 ogid=0 rdev=00:00 nametype=NORMAL cap_fp=0 cap_fi=0 cap_fe=0 cap_fver=0',

    # AVC (SELinux)
    'type=AVC msg=audit(1710494640.789:1003): avc:  denied  { read } for  pid=9876 comm="httpd" name="secret.key" dev="sda1" ino=654321 scontext=system_u:system_r:httpd_t:s0 tcontext=unconfined_u:object_r:user_home_t:s0 tclass=file permissive=0',

    # NETFILTER_PKT: сетевой пакет
    'type=NETFILTER_PKT msg=audit(1710494650.321:1004): mark=0x0 saddr=203.0.113.50 daddr=10.0.0.5 sport=44321 dport=22 proto=6',
]

In [38]:
import re
from datetime import datetime

# Специализированные Grok-шаблоны для конкретных типов auditd

# Шаблон для SYSCALL
auditd_syscall_custom = {
    'AUDIT_TYPE':    r'[A-Z_]+',
    'AUDIT_EPOCH':   r'\d+\.\d+',
    'AUDIT_SERIAL':  r'\d+',
    'HEX': r'[0-9a-fA-F]+',
}

auditd_syscall_pattern = (
    'type=SYSCALL msg=audit\(%{AUDIT_EPOCH:epoch}:%{AUDIT_SERIAL:serial}\): '
    'arch=%{HEX:arch} syscall=%{NONNEGINT:syscall:int} success=%{WORD:success} '
    'exit=%{INT:exit_code:int} '
    '%{DATA} '
    'ppid=%{NONNEGINT:ppid:int} pid=%{NONNEGINT:pid:int} '
    'auid=%{NONNEGINT:auid:int} uid=%{NONNEGINT:uid:int} gid=%{NONNEGINT:gid:int} '
    'euid=%{NONNEGINT:euid:int} '
    '%{DATA} '
    'tty=%{WORD:tty} ses=%{NONNEGINT:session:int} '
    'comm="%{DATA:command}" exe="%{DATA:executable}" key="%{DATA:audit_key}"'
)

grok_syscall = Grok(auditd_syscall_pattern, custom_patterns=auditd_syscall_custom)

# Тестируем на событии SYSCALL
result = grok_syscall.match(auditd_events[0])
print('SYSCALL event (структурированный разбор):')
pp(result)

SYSCALL event (структурированный разбор):
  epoch                = '1710494625.123'
  serial               = '1001'
  arch                 = 'c000003e'
  syscall              = 59
  success              = 'yes'
  exit_code            = 0
  ppid                 = 1234
  pid                  = 5678
  auid                 = 1000
  uid                  = 0
  gid                  = 0
  euid                 = 0
  tty                  = 'pts0'
  session              = 3
  command              = 'passwd'
  executable           = '/usr/bin/passwd'
  audit_key            = 'privileged_commands'


<>:15: SyntaxWarning: invalid escape sequence '\('
<>:15: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_3740/411943171.py:15: SyntaxWarning: invalid escape sequence '\('
  'type=SYSCALL msg=audit\(%{AUDIT_EPOCH:epoch}:%{AUDIT_SERIAL:serial}\): '


In [39]:
# Шаблон для AVC (SELinux denial)
auditd_avc_pattern = (
    'type=AVC msg=audit\(%{AUDIT_EPOCH:epoch}:%{AUDIT_SERIAL:serial}\): '
    'avc:  %{WORD:avc_result}  \{ %{DATA:permissions} \} for  '
    'pid=%{NONNEGINT:pid:int} comm="%{DATA:command}" '
    'name="%{DATA:target_name}" '
    'dev="%{DATA:device}" ino=%{NONNEGINT:inode:int} '
    'scontext=%{DATA:source_context} '
    'tcontext=%{DATA:target_context} '
    'tclass=%{WORD:target_class} permissive=%{NONNEGINT:permissive:int}'
)

grok_avc = Grok(auditd_avc_pattern, custom_patterns=auditd_custom)
result = grok_avc.match(auditd_events[4])

print('SELinux AVC denial:')
pp(result)

if result:
    print(f'\n⛔ SELinux заблокировал процесс "{result["command"]}" (PID {result["pid"]})')
    print(f'   Операция: {result["permissions"]} над файлом "{result["target_name"]}"')
    print(f'   Контекст источника: {result["source_context"]}')
    print(f'   Контекст цели:     {result["target_context"]}')

<>:3: SyntaxWarning: invalid escape sequence '\('
<>:4: SyntaxWarning: invalid escape sequence '\{'
<>:3: SyntaxWarning: invalid escape sequence '\('
<>:4: SyntaxWarning: invalid escape sequence '\{'
/tmp/ipykernel_3740/2852503504.py:3: SyntaxWarning: invalid escape sequence '\('
  'type=AVC msg=audit\(%{AUDIT_EPOCH:epoch}:%{AUDIT_SERIAL:serial}\): '
/tmp/ipykernel_3740/2852503504.py:4: SyntaxWarning: invalid escape sequence '\{'
  'avc:  %{WORD:avc_result}  \{ %{DATA:permissions} \} for  '


NameError: name 'auditd_custom' is not defined

<a id='10'></a>
## 10. Разбор событий безопасности Windows

События Windows Security Event Log в текстовом представлении (как их выводит `wevtutil` или как они отображаются в Event Viewer при экспорте) имеют многострочный формат. Рассмотрим наиболее важные для ИБ события.

Для разбора многострочных Windows-событий используются уже знакомые паттерны в новом контексте:

| Паттерн | Роль в разборе Windows Events |
|---------|-------------------------------|
| `GREEDYDATA` | Захват значений полей переменной длины (имена компьютеров, описания) |
| `NONNEGINT` | Event ID, номера портов, PID |
| `NOTSPACE` | SID, Logon ID (hex), домен, аккаунт — значения без пробелов |
| `IP` | Source Network Address |
| `SPACE` | Пропуск отступов в многострочном формате |



### 10.1 Однострочные представления Windows-событий

В ряде SIEM и при экспорте через `Get-WinEvent -Format` события могут быть представлены в однострочном формате. Для таких случаев Grok особенно удобен.

Здесь мы определяем ещё один кастомный паттерн:

| Кастомный паттерн | Regex | Описание |
|-------------------|-------|----------|
| `WIN_AUDIT_RESULT` | `Success Audit\|Failure Audit` | Результат аудита Windows (успех или провал) |

In [40]:
# Однострочные форматы, часто встречающиеся в SIEM-системах
# (Syslog-forwarded Windows events через Snare, NXLog, Winlogbeat и т.д.)

syslog_win_events = [
    # Формат Snare/NXLog для 4624
    'Mar 15 10:23:45 DC01.corp.local MSWinEventLog\t1\tSecurity\t12345\tMar 15 10:23:45\t4624\tMicrosoft-Windows-Security-Auditing\tCORP\\admin.jsmith\tN/A\tSuccess Audit\tDC01.corp.local\tLogon\tAn account was successfully logged on. Logon Type: 3 Source Network Address: 192.168.1.100 Source Port: 49152',
    # Формат для 4625 (Failed Logon)
    'Mar 15 10:24:00 DC01.corp.local MSWinEventLog\t1\tSecurity\t12346\tMar 15 10:24:00\t4625\tMicrosoft-Windows-Security-Auditing\tCORP\\svc_backup\tN/A\tFailure Audit\tDC01.corp.local\tLogon\tAn account failed to log on. Logon Type: 10 Source Network Address: 203.0.113.50 Source Port: 55555',
    # Формат для 4663 (Object Access)
    'Mar 15 10:30:15 FS01.corp.local MSWinEventLog\t1\tSecurity\t12347\tMar 15 10:30:15\t4663\tMicrosoft-Windows-Security-Auditing\tCORP\\admin.jsmith\tN/A\tSuccess Audit\tFS01.corp.local\tFile System\tAn attempt was made to access an object. Object Name: C:\\Confidential\\report.xlsx Accesses: ReadData Access Mask: 0x1',
]

# Grok-шаблон для Snare-подобного формата
win_syslog_custom = {
    'WIN_AUDIT_RESULT': r'Success Audit|Failure Audit',
}

win_syslog_pattern = (
    '%{SYSLOGTIMESTAMP:timestamp} %{HOSTNAME:host} '
    'MSWinEventLog\t%{NONNEGINT:criticality}\t%{WORD:log_name}\t'
    '%{NONNEGINT:record_id}\t%{SYSLOGTIMESTAMP:event_time}\t'
    '%{NONNEGINT:event_id}\t%{DATA:source}\t'
    '%{DATA:account}\tN/A\t%{WIN_AUDIT_RESULT:audit_result}\t'
    '%{HOSTNAME:computer}\t%{DATA:category}\t%{GREEDYDATA:message}'
)

grok_win = Grok(win_syslog_pattern, custom_patterns=win_syslog_custom)

for line in syslog_win_events:
    result = grok_win.match(line)
    if result:
        status = '✅' if result['audit_result'] == 'Success Audit' else '❌'
        print(f'{status} EventID={result["event_id"]:4s} '
              f'{result["audit_result"]:15s} '
              f'{result["account"]:25s} '
              f'{result["computer"]:20s} '
              f'{result["category"]}')

        # Дополнительно извлекаем данные из поля message
        msg = result['message']

        # Извлекаем Logon Type, Source Network Address из message
        lt = Grok('Logon Type: %{NONNEGINT:lt}').match(msg)
        sna = Grok('Source Network Address: %{IP:ip}').match(msg)
        obj = Grok('Object Name: %{GREEDYDATA:name}').match(msg)
        acc_type = Grok('Accesses: %{WORD:acc_mask}').match(msg)

        details = []
        if lt: details.append(f'LogonType={lt["lt"]}')
        if sna: details.append(f'SrcIP={sna["ip"]}')
        if obj: details.append(f'Object={obj["name"].split(" Accesses")[0]}')
        if acc_type: details.append(f'Accesses={acc_type["acc_mask"]}')
        if details:
            print(f'     Детали: {" | ".join(details)}')
    print()

✅ EventID=4624 Success Audit   CORP\admin.jsmith         DC01.corp.local      Logon
     Детали: LogonType=3 | SrcIP=192.168.1.100

❌ EventID=4625 Failure Audit   CORP\svc_backup           DC01.corp.local      Logon
     Детали: LogonType=10 | SrcIP=203.0.113.50

✅ EventID=4663 Success Audit   CORP\admin.jsmith         FS01.corp.local      File System
     Детали: Object=C:\Confidential\report.xlsx | Accesses=ReadData



<a id='11'></a>
## 11. Итоги и рекомендации

### Что мы рассмотрели

| Тема | Ключевые моменты |
|------|------------------|
| Базовый синтаксис | `%{PATTERN:field}`, `%{PATTERN:field:type}` |
| Встроенные паттерны | 100+ паттернов: IP, HOSTNAME, TIMESTAMP_ISO8601, COMBINEDAPACHELOG и др. |
| Типизация | `:int`, `:float` для автоматического приведения |
| Частичное совпадение | `match()` ищет подстроку; для полного — `^...$` |
| Кастомные паттерны | inline regex, `custom_patterns={}`, `custom_patterns_dir=` |
| Реальные форматы | Syslog, Apache, Nginx, SSH, Cisco ASA, Snort/Suricata, CEF |
| auditd | Заголовок через Grok + key=value через regex |
| Windows Events | Многострочные (посекционный разбор) + однострочные (SIEM) |

### Рекомендации по использованию

1. **Собирайте шаблон инкрементально** — добавляйте по одному `%{...}` и проверяйте `match()` на каждом шаге.

2. **Используйте `GREEDYDATA` с осторожностью** — `.*` в середине шаблона может приводить к неожиданным совпадениям и замедлению. Предпочитайте более конкретные паттерны (`DATA`, `NOTSPACE`, `WORD`).

3. **Кэшируйте объекты `Grok`** — создание объекта включает компиляцию regex. Если парсите тысячи строк одним шаблоном, создайте `Grok(...)` один раз.

4. **Для гетерогенных логов** — используйте цепочку шаблонов: пробуйте `match()` с наиболее частым шаблоном, при `None` — со следующим и т.д.

5. **Grok + Python regex = мощная комбинация** — используйте Grok для структурного разбора (заголовки, фреймы), а стандартный `re` для извлечения key=value пар из тел событий.

6. **Тестируйте шаблоны** — существуют онлайн-инструменты: [Grok Debugger](https://grokdebugger.com/), [Grok Constructor](https://grokconstructor.appspot.com/).

### Дополнительные ресурсы

| Ресурс | Описание |
|--------|----------|
| [pygrok GitHub](https://github.com/garyelephant/pygrok) | Исходный код, README, тесты |
| [Logstash Grok patterns](https://github.com/logstash-plugins/logstash-patterns-core) | Актуальная коллекция паттернов (обновляется чаще, чем в pygrok) |
| [Elastic Grok filter docs](https://www.elastic.co/guide/en/logstash/current/plugins-filters-grok.html) | Полная документация синтаксиса Grok |
| [Grok Debugger](https://grokdebugger.com/) | Онлайн-отладчик шаблонов |
| [Grok Constructor](https://grokconstructor.appspot.com/) | Построитель шаблонов с автоматическим подбором |
| [auditd Record Types](https://access.redhat.com/documentation/en-us/red_hat_enterprise_linux/7/html/security_guide/sec-audit_record_types) | Справочник типов записей auditd |
| [Windows Security Events](https://learn.microsoft.com/en-us/windows/security/threat-protection/auditing/audit-logon) | Документация Microsoft по событиям аудита |